# Chest X-Ray Model Explainability (XAI) Comparison

This notebook provides a comprehensive comparison of different explainability techniques across multiple chest X-ray classification models. 

### Techniques Included:
1. **Grad-CAM**: Visualizing regional focus for CNN and Transformer architectures.
2. **Attention Rollout**: Tracking CLS token attention through Transformer layers.
3. **SHAP (Shapley Additive Explanations)**: Holistic feature attribution.
4. **LIME (Local Interpretable Model-agnostic Explanations)**: Superpixel-based local approximations.
5. **Insertion/Deletion Evaluation**: Quantitative measurement of explanation faithfulness.

In [ ]:
# !pip install grad-cam shap lime transformers opencv-python matplotlib scikit-image tqdm

In [ ]:
import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import shap
from tqdm.notebook import tqdm
from lime import lime_image
from skimage.segmentation import mark_boundaries
from PIL import Image
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Add project root and src to path to access models and config
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(root_path, 'src')

if root_path not in sys.path:
    sys.path.append(root_path)
if src_path not in sys.path:
    sys.path.append(src_path)

import config
from model import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# --- configuration ---
MODEL_WEIGHTS = {
    "raddino": "../model_checkpoints/raddino_best_model.pth",
    "radjepa": "../model_checkpoints/radjepa_best_model.pth",
    "swin": "../model_checkpoints/swin_best_model.pth",
    "efficientnet": "../model_checkpoints/efficientnet_best_model.pth",
    "convnext": "../model_checkpoints/convnext_best_model.pth",
    "cnn_transformer": "../model_checkpoints/cnn_transformer_best_model.pth",
    "resnet50": "../model_checkpoints/resnet50_best_model.pth"
}

TEST_IMAGES = [
    "../dataset/00000001_002.png",
    "../dataset/00000003_003.png",
    "../dataset/00000005_007.png",
    "../dataset/00000008_001.png",
    "../dataset/00000008_002.png"
]

# Utility functions for Grad-CAM with Transformers
def reshape_transform_vit(tensor):
    n_tokens = tensor.size(1)
    if int(np.sqrt(n_tokens - 1))**2 == n_tokens - 1:
        grid_size = int(np.sqrt(n_tokens - 1))
        result = tensor[:, 1:, :].reshape(tensor.size(0), grid_size, grid_size, tensor.size(2))
    else:
        grid_size = int(np.sqrt(n_tokens))
        result = tensor.reshape(tensor.size(0), grid_size, grid_size, tensor.size(2))
    
    result = result.permute(0, 3, 1, 2)
    return result

def reshape_transform_h_w_c(tensor):
    return tensor.permute(0, 3, 1, 2)

In [ ]:
def load_trained_model(model_name):
    original_name = config.MODEL_NAME
    config.MODEL_NAME = model_name
    
    try:
        model = get_model()
        weights_path = MODEL_WEIGHTS.get(model_name)
        if weights_path and os.path.exists(weights_path):
            model.load_state_dict(torch.load(weights_path, map_location=device))
        else:
            print(f"Warning: No weights found for {model_name} at {weights_path}.")
        
        # Unfreeze all parameters for XAI analysis (Grad-CAM requires gradients)
        for param in model.parameters():
            param.requires_grad = True
            
        model.to(device)
        model.eval()
        return model
    finally:
        config.MODEL_NAME = original_name

def get_preprocess_transform():
    return transforms.Compose([
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_image_tensor(image_path):
    img = Image.open(image_path).convert('RGB')
    transform = get_preprocess_transform()
    return transform(img).unsqueeze(0).to(device)

## 1. Grad-CAM Comparative Visualization

In [ ]:
def generate_all_gradcam(image_path, model_names):
    num_models = len(model_names)
    fig, axes = plt.subplots(1, num_models + 1, figsize=(5 * (num_models + 1), 5))
    
    img = Image.open(image_path).convert('RGB')
    axes[0].imshow(img.resize((config.IMAGE_SIZE, config.IMAGE_SIZE)))
    axes[0].set_title("Original X-Ray")
    axes[0].axis('off')
    
    input_tensor = get_image_tensor(image_path)
    rgb_img = np.float32(img.resize((config.IMAGE_SIZE, config.IMAGE_SIZE))) / 255

    for i, model_name in enumerate(tqdm(model_names, desc="Grad-CAM Models")):
        # Handle models with different input size requirements
        current_size = 384 if model_name == "swin" else config.IMAGE_SIZE
        model = load_trained_model(model_name)
        input_tensor = get_image_tensor(image_path, size=current_size)
        rgb_img_model = np.float32(img.resize((current_size, current_size))) / 255
        
        reshape_transform = None
        if model_name == "efficientnet":
            target_layers = [model.model.conv_head]
        elif model_name == "convnext":
            target_layers = [model.model.stages[-1].blocks[-1]]
        elif model_name == "resnet50":
            target_layers = [model.backbone.layer4[-1]] 
        elif model_name == "raddino":
            target_layers = [model.encoder.encoder.layer[-1].norm1]
            reshape_transform = reshape_transform_vit
        elif model_name == "radjepa":
            target_layers = [model.encoder.model.blocks[-1].norm1]
            reshape_transform = reshape_transform_vit
        elif model_name == "swin":
            target_layers = [model.model.layers[-1].blocks[-1]]
            reshape_transform = reshape_transform_h_w_c
        elif model_name == "cnn_transformer":
            target_layers = [model.conv4[-1]]
        else:
            continue

        cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape_transform)
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
        cam_image = show_cam_on_image(rgb_img_model, grayscale_cam, use_rgb=True)
        
        axes[i+1].imshow(cam_image)
        axes[i+1].set_title(f"Grad-CAM: {model_name}")
        axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()

## 2. Attention Rollout Comparison (Transformers)

In [ ]:
def compute_rollout(attentions):
    result = torch.eye(attentions[0].size(-1)).to(attentions[0].device)
    for attention in attentions:
        attention_heads_fused = attention.mean(axis=1)
        attention_heads_fused += torch.eye(attention_heads_fused.size(-1)).to(attention.device)
        attention_heads_fused = attention_heads_fused / attention_heads_fused.sum(dim=-1, keepdim=True)
        result = torch.matmul(attention_heads_fused, result)
    return result

def generate_all_rollout(image_path, transformer_names):
    num_models = len(transformer_names)
    fig, axes = plt.subplots(1, num_models + 1, figsize=(5 * (num_models + 1), 5))
    
    img = Image.open(image_path).convert('RGB')
    resized_img = img.resize((config.IMAGE_SIZE, config.IMAGE_SIZE))
    axes[0].imshow(resized_img)
    axes[0].set_title("Original X-Ray")
    axes[0].axis('off')
    
    input_tensor = get_image_tensor(image_path)

    for i, model_name in enumerate(tqdm(transformer_names, desc="Rollout Models")):
        model = load_trained_model(model_name)
        
        with torch.no_grad():
            if hasattr(model, 'encoder') and hasattr(model.encoder, 'forward'):
                outputs = model.encoder(input_tensor, output_attentions=True)
                if hasattr(outputs, 'attentions'):
                    attentions = outputs.attentions
                    rollout = compute_rollout(attentions)
                    cls_attention = rollout[0, 0, 1:]
                    grid_size = int(np.sqrt(cls_attention.size(0)))
                    attention_map = cls_attention.reshape(grid_size, grid_size).cpu().numpy()
                    heatmap = cv2.resize(attention_map, (config.IMAGE_SIZE, config.IMAGE_SIZE))
                    
                    axes[i+1].imshow(heatmap, cmap='jet')
                    axes[i+1].set_title(f"Rollout: {model_name}")
                    axes[i+1].axis('off')
                    continue
            
            axes[i+1].text(0.5, 0.5, 'N/A', ha='center', va='center')
            axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()

## 3. LIME Comparison

In [ ]:
def run_comparative_lime(image_path, model_names):
    num_models = len(model_names)
    fig, axes = plt.subplots(1, num_models, figsize=(5 * num_models, 5))
    
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img.resize((config.IMAGE_SIZE, config.IMAGE_SIZE)))
    transform = get_preprocess_transform()

    explainer = lime_image.LimeImageExplainer()

    for i, model_name in enumerate(tqdm(model_names, desc="LIME Models")):
        model = load_trained_model(model_name)
        
        def batch_predict(images):
            batch = torch.stack([transform(Image.fromarray(i)) for i in images]).to(device)
            with torch.no_grad():
                return torch.sigmoid(model(batch)).cpu().numpy()

        explanation = explainer.explain_instance(img_array, batch_predict, top_labels=1, num_samples=500)
        temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False)
        
        img_boundary = mark_boundaries(temp/255.0, mask)
        axes[i].imshow(img_boundary)
        axes[i].set_title(f"LIME: {model_name}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## 4. Faithfulness Evaluation (Insertion/Deletion)

In [ ]:
def calculate_metrics(model, image_path, heatmap, steps=10):
    transform = get_preprocess_transform()
    img = Image.open(image_path).convert('RGB')
    resized_image = np.array(img.resize((config.IMAGE_SIZE, config.IMAGE_SIZE)))
    
    flat_heatmap = heatmap.flatten()
    sorted_indices = np.argsort(flat_heatmap)[::-1]
    total_pixels = len(sorted_indices)
    
    mean_color = np.array([0.485, 0.456, 0.406]) * 255
    
    fractions = np.linspace(0, 1.0, steps + 1)
    deletion_scores = []
    insertion_scores = []

    base_tensor = transform(Image.fromarray(resized_image)).unsqueeze(0).to(device)
    with torch.no_grad():
        base_prob = torch.sigmoid(model(base_tensor))[0]
        target_class = torch.argmax(base_prob).item()

    with torch.no_grad():
        for frac in tqdm(fractions, desc="Ins/Del Steps", leave=False):
            num_pixels = int(frac * total_pixels)
            important_pixels = sorted_indices[:num_pixels]
            
            del_img = resized_image.copy().reshape(-1, 3)
            del_img[important_pixels] = mean_color
            del_tensor = transform(Image.fromarray(del_img.reshape(config.IMAGE_SIZE, config.IMAGE_SIZE, 3))).unsqueeze(0).to(device)
            deletion_scores.append(torch.sigmoid(model(del_tensor))[0, target_class].item())
            
            ins_img = np.full_like(resized_image, mean_color).reshape(-1, 3)
            ins_img[important_pixels] = resized_image.reshape(-1, 3)[important_pixels]
            ins_tensor = transform(Image.fromarray(ins_img.reshape(config.IMAGE_SIZE, config.IMAGE_SIZE, 3))).unsqueeze(0).to(device)
            insertion_scores.append(torch.sigmoid(model(ins_tensor))[0, target_class].item())
            
    return fractions, deletion_scores, insertion_scores

def run_comparative_eval(image_path, model_names):
    plt.figure(figsize=(10, 6))
    
    for model_name in tqdm(model_names, desc="Faithfulness Eval"):
        model = load_trained_model(model_name)
        dummy_heatmap = np.random.rand(config.IMAGE_SIZE, config.IMAGE_SIZE)
        
        fracs, del_s, ins_s = calculate_metrics(model, image_path, dummy_heatmap)
        
        plt.plot(fracs * 100, del_s, label=f'{model_name} (Del)')
        plt.plot(fracs * 100, ins_s, linestyle='--', label=f'{model_name} (Ins)')
    
    plt.title("Insertion/Deletion Comparison Across Models")
    plt.xlabel("Percentage of Pixels Modified")
    plt.ylabel("Model Confidence")
    plt.legend()
    plt.grid(True)
    plt.show()

## 5. Execution

In [ ]:
all_models = ["raddino", "radjepa", "swin", "efficientnet", "convnext", "cnn_transformer", "resnet50"]
transformer_models = ["raddino", "radjepa", "swin", "cnn_transformer"]
test_image = TEST_IMAGES[0]

generate_all_gradcam(test_image, all_models)
generate_all_rollout(test_image, transformer_models)
run_comparative_lime(test_image, all_models)
run_comparative_eval(test_image, all_models)